In [ ]:
#===============================
# USER CONFIGURATION
# ===============================

# Name of the Excel file exported from the plate reader
# The first row must contain time points (hours)
# The first column must contain experiment names (e.g. REF_A, REF_B, COND1_A ...)
file_name = "20250418_Pretreated_Results_0.1nM_mCherry.xlsx" # name of  data file

# Full path to the input dataset
file_path = f'{file_name}' 

# Set the base output directory where the results folder will be created
base_output_dir = r""  # <-- Change this to your desired location

# ===============================
# PLOT APPEARANCE
# ===============================

# Colors used for mean and replicate curves
mean_curve_color = 'black'
curve_color_1 = 'blueviolet'
curve_color_2 = 'blue'
curve_color_3 = 'limegreen'


# If True → all plots share identical axis limits
# Useful when visually comparing multiple conditions
NORMALISATION = False

# ===============================
# ANALYSIS PARAMETERS
# ===============================

# Choice of reference condition used to compute relative yields
# "REF" → internal reference PURE condition
# "COM" → commercial PURE system
relative_yield_choice = "REF" # REF ou COM

# Day label (used for optional plot titles)
day = "1" 



In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import os
from scipy.optimize import curve_fit
import random
from matplotlib.ticker import ScalarFormatter
import concurrent.futures
import sys

# ---------------------------------------------------------------------------
# Internal utility: figure scaling helper
# ---------------------------------------------------------------------------
# Adds the lab's shared Python folder to the import path so that figure_utils
# can be found without installing it as a package.
sys.path.append(r"") # Path to folder containing figure_utils.py
from figure_utils import set_scaled_rcparams   # sets rcParams and returns layout constants


# ======================================
# FIGURE STYLE INITIALIZATION
# ======================================

# Target figure size for all plots (inches)
target_figsize = (2, 1.8)

# Automatically scale matplotlib parameters for consistent figure layout
font_family, scale, marker_size, labelpad, tight_pad = set_scaled_rcparams(target_figsize)  

def create_directory(dir_name):
    """Create a directory if it does not exist."""
    if not os.path.exists(dir_name):
        os.makedirs(dir_name)
        print(f"Directory {dir_name} created.")
    else:
        print(f"Directory {dir_name} already exists.")

def initialize_directories(base_output_dir=None):
    """
    Create the output folder structure for a given experiment.

    Structure:
        Day_X_kinetic_plots/
            all_curves/
            mean_curves/
            fitted_curves/

    Returns
    -------
    dir_path : str
        Path to the main experiment directory.
    """
    dir_name = f"Day_{day}_kinetic_plots"
    if base_output_dir is not None:
        dir_path = os.path.join(base_output_dir, dir_name)
    else:
        dir_path = dir_name
    sub_dirs = ["all_curves", "mean_curves", "fitted_curves"]
    create_directory(dir_path)
    for sub_dir in sub_dirs:
        create_directory(os.path.join(dir_path, sub_dir))
    return dir_path

def process_data(df):
    """
    Extract time points and experiment identifiers from the raw dataset.

    Assumptions about the Excel format:
        - Row 0 contains time values (hours)
        - Column 0 contains experiment names

    Returns
    -------
    hours : numpy array
        Measurement time points

    experiences_names : pandas Series
        Experiment identifiers
    """
    hours = df.iloc[0, 1:].astype(float)
    experiences_names = df.iloc[:, 0]
    experiences_names = experiences_names.fillna('MISSING').astype(str)
    return hours, experiences_names

def group_replicates(experiences_names, df):
    """
    Group replicate measurements by condition prefix.

    Example:
        REF_A, REF_B → group "REF"
        COND1_A, COND1_B → group "COND1"

    Replicate suffixes must be one of:
        A, B, C, D, E

    Returns
    -------
    replicates : dict
        {
            "REF": [(REF_A, values), (REF_B, values)],
            "COND1": [(COND1_A, values), ...]
        }
    """

    replicates = {}
    for i, exp in enumerate(experiences_names):
        if exp.endswith(('A', 'B', 'C', 'D', 'E')):
            exp_prefix = exp[:-1]
            if exp_prefix not in replicates:
                replicates[exp_prefix] = []
            replicates[exp_prefix].append((exp, df.iloc[i, 1:].astype(float).values))
    return replicates

def calculate_mean(tuples_list):
    """
    Compute the mean fluorescence curve of replicates (tuples).
    """
    values = np.array([t[1] for t in tuples_list])
    return np.mean(values, axis=0)

def calculate_std(tuples_list):
    """Calculate the standard deviation of a list of replicates/tuples."""
    values = np.array([t[1] for t in tuples_list])
    return np.std(values, axis=0)

def get_sliding_window_max_yield(values, window_size=5):
    """
    Calculate the maximum average of a sliding window over the values.
    This helps avoid noise by considering a range of points.
     Parameters
    ----------
    values : array
        Fluorescence values
    window_size : int
        Number of points used for smoothing

    Returns
    -------
    float
        Maximum window average

    """
    max_average = -np.inf
    for i in range(len(values) - window_size + 1):
        window = values[i:i + window_size]
        window_average = np.mean(window)
        if window_average > max_average:
            max_average = window_average
    return max_average


def get_yield(value, ref, window_size=5):
    """
    Calculate the relative yield using the sliding window maximum.
    Compute relative protein expression yield.

    Relative yield = (max sliding window of condition) /
            (max sliding window of reference)

    Returns
    -------
    float
    """
    max_value = get_sliding_window_max_yield(value, window_size)
    max_ref = get_sliding_window_max_yield(ref, window_size)
    return max_value / max_ref


def plot_and_save_curves_ALL(replicates, hours, ref_mean, dir_name, NORMALISATION, all_mean_curve, com_curve, fmt_config):
    """Plot and save curves for each experiment prefix."""

    for exp_prefix, values in replicates.items():
        plt.figure(figsize=target_figsize, dpi=300)
        cpt = 0
        for exp_name, value in values:
            if cpt == 0:
                curve_color = curve_color_1
            if cpt == 1:
                curve_color = curve_color_2
            if cpt == 2:
                curve_color = curve_color_3
            cpt += 1

            plt.plot(hours, value, label=exp_name, color=curve_color)

        mean_values = calculate_mean(values)
        
        # Append "COM" to com_curve, but exclude "REF"
        if exp_prefix == "COM":
            com_curve.append({'name': exp_prefix, 'values': mean_values})
        elif exp_prefix != "REF":
            all_mean_curve.append({'name': exp_prefix, 'values': mean_values})

        exp_yield = get_yield(mean_values, ref_mean, window_size=5)
        print("Rel. Yields:", exp_yield)
        
        if len(values) > 1:
            plt.plot(hours, mean_values, label=f'{exp_prefix} Mean', linestyle='--', color=mean_curve_color)

        plt.xlabel('Hour')
        plt.ylabel('Value')
        #plt.suptitle(f'Day {day}', fontsize = 10)
        #plt.title(f'Cond. : {exp_prefix} | Rel. Yield : {round(exp_yield, 2)}')
        # Enforce scientific notation for both axes
        formatter = ScalarFormatter(useOffset=False)
        formatter.set_scientific(True)
        formatter.set_powerlimits((0, 0))  # Force scientific notation for all non-zero numbers
        formatter.format = '%1.1e'

        plt.gca().yaxis.set_major_formatter(formatter)
        plt.legend()

        if NORMALISATION:
            all_values = np.concatenate([data for sublist in replicates.values() for _, data in sublist])
            min_y = np.min(all_values)
            max_y = np.max(all_values)
            min_x = np.min(hours)
            max_x = np.max(hours)
            plt.xlim(min_x, max_x)
            plt.ylim(min_y, max_y)

        save_path = os.path.join(dir_name, "all_curves", f"{exp_prefix}_curve.png")
        plt.tight_layout()
        plt.savefig(save_path, bbox_inches='tight')
        # plt.show()
        plt.close()

    return all_mean_curve, com_curve


def plot_and_save_curves_MEAN(replicates, hours, ref_mean, dir_name, NORMALISATION):
    """Plot and save mean curves with standard deviation for each experiment prefix."""

    # Create an empty DataFrame to store statistics
    stats_df = pd.DataFrame(columns=["Condition", "Relative Yield", "Last Std Dev", "Corresponding Mean", "Percentage"])
    
    for exp_prefix, values in replicates.items():
        plt.figure(figsize=target_figsize, dpi=300)

        # Calculate the mean values for this experiment
        mean_values = calculate_mean(values)
        std_values = None
        if len(values) > 1:
            # Calculate standard deviation if there are more than one replicate
            std_values = calculate_std(values)
            plt.fill_between(hours, mean_values - std_values, mean_values + std_values, color='gray', alpha=0.5)

        # Print out the last standard deviation point as a percentage of the corresponding mean point
        last_std_point, corresponding_mean_point, percentage = None, None, None
        if std_values is not None:
            last_std_point = std_values[-1]
            corresponding_mean_point = mean_values[-1]
            percentage = (last_std_point / corresponding_mean_point) * 100
            print(f"Condition: {exp_prefix} | Last Std Dev: {last_std_point:.2f} | Corresponding Mean: {corresponding_mean_point:.2f} | Percentage: {percentage:.2f}")

        # Calculate yield
        exp_yield = get_yield(mean_values, ref_mean, window_size=5)

        # Add the data to the DataFrame
        new_row = {
            "Condition": exp_prefix,
            "Last Std Dev": last_std_point,
            "Corresponding Mean": corresponding_mean_point,
            "Percentage": percentage,
            "Relative Yield": exp_yield
        }
        stats_df = pd.concat([stats_df, pd.DataFrame([new_row])], ignore_index=True)

        # Plot the mean values
        plt.plot(hours, mean_values, label=f'{exp_prefix} Mean', linestyle='--', color='black')

        # Add labels and titles
        plt.xlabel('Hour')
        plt.ylabel('Value')
        #plt.suptitle(f'Day {day}', fontsize=10)
        #plt.title(f'Cond. : {exp_prefix} | Rel. Yield : {round(exp_yield, 2)}')

        # Set scientific notation for axes
        formatter = ScalarFormatter(useOffset=False)
        formatter.set_scientific(True)
        formatter.set_powerlimits((0, 0))  # Force scientific notation for all non-zero numbers
        formatter.format = '%1.1e'

        plt.gca().yaxis.set_major_formatter(formatter)
        plt.legend()

        # Normalization (optional)
        if NORMALISATION:
            all_values = np.concatenate([data for sublist in replicates.values() for _, data in sublist])
            min_y = np.min(all_values)
            max_y = np.max(all_values)
            min_x = np.min(hours)
            max_x = np.max(hours)
            plt.xlim(min_x, max_x)
            plt.ylim(min_y, max_y)

        # Save the plot
        save_path = os.path.join(dir_name, "mean_curves", f"{exp_prefix}_mean_curve.png")
        plt.tight_layout()
        plt.savefig(save_path, bbox_inches='tight')
        plt.close()

    # Save the statistics DataFrame to an Excel file
    stats_excel_path = os.path.join(dir_name, "statistics_summary.xlsx")
    stats_df.to_excel(stats_excel_path, index=False)

   


# Sigmoid model function
def sigmoid(t, k_prime, k, K, n):
    """
    Four-parameter Hill-type sigmoid model.

    Parameters
    ----------
    k_prime : baseline fluorescence
    k : maximal yield
    K : half-time constant
    n : Hill coefficient (curve steepness)
    """

    return k_prime + (k * t**n) / (t**n + K**n)

# Function to calculate Plateau_Time
def calculate_Plateau_Time(K, n):
    
    return ((2 * K)/n) + K

# Function to calculate apparent Translation_Rate
def calculate_Translation_Rate(k, n, K):
    """
    Estimate apparent protein production rate
    from sigmoid parameters.
    """

    return k * n / (4 * K)

def fit_with_timeout(time_points, fluorescence_values, initial_guesses, maxfev=100000, timeout=10):
    """
    Run nonlinear regression with a timeout safeguard.
    """
    def fitting_task():
        return curve_fit(sigmoid, time_points, fluorescence_values, p0=initial_guesses, maxfev=maxfev)

    with concurrent.futures.ThreadPoolExecutor() as executor:
        future = executor.submit(fitting_task)
        try:
            return future.result(timeout=timeout)  # Wait for the result with a timeout
        except concurrent.futures.TimeoutError:
            print("Fitting process timed out.")
            return None, None

def fit_with_retries(time_points, fluorescence_values, max_retries=1, timeout=20,):
    """
    Retry the sigmoid fitting process multiple times with different initial guesses.
    If all retries fail, return None and move on to the next entry.
    """
    for retry in range(max_retries):
        # Generate initial guesses
        k_prime = np.percentile(fluorescence_values, 5)  # Lower percentile for k_prime
        k = np.percentile(fluorescence_values, 95) - k_prime  # Range for k
        K = np.median(time_points)  # Median as a starting point for K
        n = 2  # Fixed initial value for n
        initial_guesses = [k_prime, k, K, n]

        # Attempt to fit with a timeout
        popt, pcov = fit_with_timeout(time_points, fluorescence_values, initial_guesses, timeout=timeout)
        if popt is not None:
            return popt, pcov
        else:
            print(f"Retry {retry + 1}/{max_retries} failed for current entry.")

    # If all retries fail, log a warning and return None
    print("WARNING: All retries failed for the current entry. Moving to the next entry.")
    return None, None

def plot_and_save_curves_SIGMOID(replicates, hours, ref_mean, dir_name, NORMALISATION, redo=False):
    """
    Plot and save individual curves with fitted sigmoid, retrying the fitting process if necessary.
    """
    results = []
    for _, values in replicates.items():
        for exp_name, value in values:
            plt.figure()
            plt.plot(hours, value, label=exp_name, color="blue")

            if redo:
                for i in range(20):
                    adjustment = random.uniform(-5, 5)
                    value[i] += adjustment

            # Retry fitting multiple times
            popt, pcov = fit_with_retries(hours, value)

            if popt is not None:
                # Extract parameters
                k_prime, k, K, n = popt
                fitted_values = sigmoid(hours, *popt)

                # Calculate additional metrics
                Plateau_Time = calculate_Plateau_Time(K, n)
                Translation_Rate = calculate_Translation_Rate(k, n, K)
                exp_yield = get_yield(fitted_values, ref_mean)

                # Plot fitted sigmoid curve
                plt.plot(hours, fitted_values, label="Fitted Sigmoid", linestyle="--", color="red")
                #plt.suptitle(f"Day {day}", fontsize=10)
                #plt.title(
                    #f"Cond. : {exp_name} Rel. Yield : {round(exp_yield, 2)}\n"
                    #f"k_prime={k_prime:.2f}, Yield={k:.2f}, K={K:.2f}, n={n:.2f}, "
                    #f"plateau_t={Plateau_Time:.2f}, t_r={Translation_Rate:.2f}",
                    #fontsize=10,
                #)
                plt.subplots_adjust(top=0.83)

                # Save results
                results.append({
                    "Cond.": exp_name,
                    "k_prime": k_prime,
                    "Yield": k,
                    "K": K,
                    "n": n,
                    "Plateau_Time": Plateau_Time,
                    "Translation_Rate": Translation_Rate,
                    "Rel. Yield": exp_yield,
                })
            else:
                print(f"WARNING: Fitting failed for {exp_name}. Plotting only experimental data.")

            # Plot settings
            plt.xlabel("Hour")
            plt.ylabel("Value")
            plt.legend()

            if NORMALISATION:
                all_values = np.concatenate([data for sublist in replicates.values() for _, data in sublist])
                min_y = np.min(all_values)
                max_y = np.max(all_values)
                min_x = np.min(hours)
                max_x = np.max(hours)
                plt.xlim(min_x, max_x)
                plt.ylim(min_y, max_y)

            save_path = os.path.join(dir_name, "fitted_curves", f"{exp_name}_sigmoid_curve.png")
            plt.tight_layout()
            plt.savefig(save_path, bbox_inches='tight')
            plt.close()

    # Save results to a CSV file
    results_df = pd.DataFrame(results)
    csv_path = os.path.join(dir_name, "results.csv")
    if redo:
        old_df = pd.read_csv(csv_path)
        results_df = pd.concat([old_df, results_df], ignore_index=True)
    results_df.to_csv(csv_path, index=False)

    
def plot_and_save_all_mean_curve(all_mean_curve, com_curve, relative_choice_curve, hours, ref_mean, dir_name, fmt_config):
    """
    Plot all mean curves together, highlighting the COM condition in cyan.
    Y-axis will show scientific notation in the label (RFU·10^X).
    """
    plt.figure(figsize=target_figsize, dpi=300)

    # Compute the global max to determine order of magnitude
    all_values = []
    for mean_data in all_mean_curve:
        all_values.extend(mean_data['values'])
    all_values.extend(calculate_mean(relative_choice_curve))  # include reference
    all_values = np.array(all_values)
    
    # Determine order of magnitude
    max_val = np.max(np.abs(all_values))
    if max_val == 0:
        order_mag = 0
    else:
        order_mag = int(np.floor(np.log10(max_val)))
    
    # Scaling factor
    scale = 10**order_mag

    # Plot all other conditions with green or red based on yield
    for mean_data in all_mean_curve:
        exp_yield = get_yield(mean_data['values'], ref_mean, window_size=5)
        y_scaled = np.array(mean_data['values']) / scale
        color = 'green' if exp_yield > 1 else 'red'
        plt.plot(hours, y_scaled, color=color, label=mean_data['name'], linewidth=0.65)

    # Plot the reference curve
    ref = np.array(calculate_mean(relative_choice_curve)) / scale
    plt.plot(hours, ref, color='black', label='Reference (REF)', linewidth=0.65)

    # X and Y labels
    plt.xlabel('Time (h)', labelpad=labelpad)
    plt.ylabel(f'Fluorescence ×10$^{order_mag}$ (RFU)', labelpad=labelpad)

    # Optionally plot COM curve
    # for com_data in com_curve:
    #     plt.plot(hours, np.array(com_data['values']) / scale, color='cyan', label='COM')

    # Enforce plain formatting for y-axis ticks (values are already scaled)
    plt.gca().yaxis.set_major_formatter(ScalarFormatter(useOffset=False))
    
    save_path_png = os.path.join(dir_name, "all_mean_and_com_curve_plot_.png")
    save_path_svg = os.path.join(dir_name, "all_mean_and_com_curve_plot_.svg")

    plt.tight_layout(pad=tight_pad)
    plt.savefig(save_path_png, bbox_inches='tight')  # save as PNG
    plt.savefig(save_path_svg, bbox_inches='tight')  # save as SVG
    plt.close()



def plot_from_results_csv(csv_path):
    results_df = pd.read_csv(csv_path)

    correlation_dir = os.path.join(os.path.dirname(csv_path), "correlations")
    create_directory(correlation_dir)

    plt.figure()
    plt.scatter(results_df['Yield'], results_df['Plateau_Time'], label='Plateau_Time vs Yield', color='blue')
    plt.xlabel('Yield (RFU)')
    plt.ylabel('Plateau_Time (h)')
    #plt.suptitle(f'Day {day}', fontsize = 10)
    #plt.title('Plateau_Time vs Yield')
    # Enforce scientific notation for both axes
    formatter = ScalarFormatter(useOffset=False)
    formatter.set_scientific(True)
    formatter.set_powerlimits((0, 0))  # Force scientific notation for all non-zero numbers
    formatter.format = '%1.1e'

    plt.gca().xaxis.set_major_formatter(formatter)
    
    plt.tight_layout()
    plt.savefig(os.path.join(correlation_dir, 'Plateau_Time vs Yield.png'), bbox_inches='tight')
    plt.close()

    plt.figure()
    plt.scatter(results_df['Yield'], results_df['Translation_Rate'], label='Translation_Rate vs Yield', color='green')
    plt.xlabel('Yield (RFU)')
    plt.ylabel('Translation_Rate (RFU.min^-1)')
    #plt.suptitle(f'Day {day}', fontsize = 10)
    #plt.title('Translation Rates vs Yields (All Replicates)')
    # Enforce scientific notation for both axes
    formatterX = ScalarFormatter(useOffset=False)
    formatterX.set_scientific(True)
    formatterX.set_powerlimits((0, 0))  # Force scientific notation for all non-zero numbers
    formatterX.format = '%1.1e'
    formatterY = ScalarFormatter(useOffset=False)
    formatterY.set_scientific(True)
    formatterY.set_powerlimits((0, 0))  # Force scientific notation for all non-zero numbers
    formatterY.format = '%1.1e'
    plt.gca().xaxis.set_major_formatter(formatterX)
    plt.gca().yaxis.set_major_formatter(formatterY)

    plt.tight_layout()
    plt.savefig(os.path.join(correlation_dir, 'Translation_Rate_vs Yield.png'), bbox_inches='tight')
    plt.close()

    plt.figure()
    plt.scatter(results_df['Plateau_Time'], results_df['Translation_Rate'], label='Plateau_Time vs Translation_Rate', color='red')
    plt.xlabel('Plateau_Time (h)')
    plt.ylabel('Translation_Rate (RFU.min^-1)')
    #plt.suptitle(f'Day {day}', fontsize = 10)
    #plt.title('Plateau_Time vs Translation_Rate')

    # Enforce scientific notation for both axes
    formatter = ScalarFormatter(useOffset=False)
    formatter.set_scientific(True)
    formatter.set_powerlimits((0, 0))  # Force scientific notation for all non-zero numbers
    formatter.format = '%1.1e'

    
    plt.gca().yaxis.set_major_formatter(formatter)
    plt.tight_layout()
    plt.savefig(os.path.join(correlation_dir, 'Plateau_Time_vs_Translation_Rate.png'), bbox_inches='tight')
    plt.close()

def main():

    # ==============================
    # DATA LOADING
    # ==============================


    # Initialize directories and read data
    dir_name = initialize_directories(file_name, base_output_dir)
    df = pd.read_excel(file_path)
    hours, experiences_names = process_data(df)
    replicates = group_replicates(experiences_names, df)



    # ==============================
    # REFERENCE SELECTION
    # ==============================

    # Determine reference curve based on relative yield choice
    if relative_yield_choice == "REF": 
        relative_choice_curve = replicates["REF"]
        ref_mean = calculate_mean(replicates["REF"])
    elif relative_yield_choice == "COM":
        relative_choice_curve = replicates["COM"]
        ref_mean = calculate_mean(replicates["COM"])

    
    # ==============================
    # MEAN CURVE STORAGE
    # ==============================

    # Initialize lists to hold mean curves
    all_mean_curve = []
    com_curve = []


    # ==============================
    # FORMAT CONFIGURATION
    # ==============================

    # Get the scaling parameters
    params = set_scaled_rcparams(target_figsize)
    # Bundle them for easy passing
    fmt_config = {
        'font': params[0],
        'scale': params[1],
        'marker_size': params[2],
        'labelpad': params[3],
        'tight_pad': params[4]
    }

    # ==============================
    # PLOT INDIVIDUAL CURVES
    # ==============================

    # Pass 'fmt_config' to your plotting functions
    all_mean_curve, com_curve = plot_and_save_curves_ALL(
        replicates, hours, ref_mean, dir_name, NORMALISATION, all_mean_curve, com_curve, fmt_config
    )
    
    
    # ==============================
    # GLOBAL MEAN CURVE FIGURE
    # ==============================

    # Do the same for other functions...
    plot_and_save_all_mean_curve(all_mean_curve, com_curve, relative_choice_curve, hours, ref_mean, dir_name, fmt_config)

    # ==============================
    # CORRELATION ANALYSIS
    # ==============================

    # Plot from results CSV
    csv_path = os.path.join(dir_name, "results.csv")
    plot_from_results_csv(csv_path)


if __name__ == "__main__":
    main()

    
        

Directory U:\\Thèse\\Data\\Data analysis\\Echo PURE SynChro\\Article\20250623_Pretreated_Results_1nM_mCherry already exists.
Directory U:\\Thèse\\Data\\Data analysis\\Echo PURE SynChro\\Article\20250623_Pretreated_Results_1nM_mCherry\all_curves already exists.
Directory U:\\Thèse\\Data\\Data analysis\\Echo PURE SynChro\\Article\20250623_Pretreated_Results_1nM_mCherry\mean_curves already exists.
Directory U:\\Thèse\\Data\\Data analysis\\Echo PURE SynChro\\Article\20250623_Pretreated_Results_1nM_mCherry\fitted_curves already exists.
Rel. Yields: 3.04031699889258
Rel. Yields: 2.0543673864894796
Rel. Yields: 3.75
Rel. Yields: 3.560112126245847
Rel. Yields: 2.5372715946843853
Rel. Yields: 3.169089147286822
Rel. Yields: 3.61593300110742
Rel. Yields: 0.7088178294573643
Rel. Yields: 1.3992248062015504
Rel. Yields: 1.0904277408637875
Rel. Yields: 2.447916666666667
Rel. Yields: 3.098318106312292
Rel. Yields: 1.2054263565891472
Rel. Yields: 2.111157253599114
Rel. Yields: 2.211932447397564
Rel. Yi

In [ ]:
# =====================================
# MANUAL SIGMOID REFITTING TOOL
# =====================================

# Reload data


dir_name = initialize_directories(file_name, base_output_dir)
df = pd.read_excel(file_path)
hours, experiences_names = process_data(df)
replicates = group_replicates(experiences_names, df)
ref_mean = calculate_mean(replicates["REF"])
NORMALISATION= True

def filter_replicates(replicates, exp_to_include):
    """
    Select specific replicates for manual re-analysis.

    Parameters
    ----------
    exp_to_include : list
        Names of experiments to refit
    """
    filtered_replicates = {}
    for exp_prefix, values in replicates.items():
        filtered_values = [value for value in values if value[0] in exp_to_include]
        if filtered_values:
            filtered_replicates[exp_prefix] = filtered_values
    return filtered_replicates

exp_to_redo = input("Enter the experiment names to redo (separated by commas): ").strip().split(',')
exp_to_redo = [exp.strip() for exp in exp_to_redo]

filtered_replicates = filter_replicates(replicates, exp_to_redo)
plot_and_save_curves_SIGMOID(filtered_replicates, hours, ref_mean, dir_name, NORMALISATION, True)

Directory U:\\Thèse\\Data\\Data analysis\\Echo PURE SynChro\\Article\20250623_Pretreated_Results_1nM_mCherry already exists.
Directory U:\\Thèse\\Data\\Data analysis\\Echo PURE SynChro\\Article\20250623_Pretreated_Results_1nM_mCherry\all_curves already exists.
Directory U:\\Thèse\\Data\\Data analysis\\Echo PURE SynChro\\Article\20250623_Pretreated_Results_1nM_mCherry\mean_curves already exists.
Directory U:\\Thèse\\Data\\Data analysis\\Echo PURE SynChro\\Article\20250623_Pretreated_Results_1nM_mCherry\fitted_curves already exists.
